In [ ]:
import numpy as np
import pandas as pd

movies = pd.read_csv('Data/tmdb_5000_movies.csv')
credits = pd.read_csv('Data/tmdb_5000_credits.csv')

movies = movies.merge(credits,on='title') 

#Figuring out the columns that are important
#1.genres
#2.id
#3.Keywords
#4.title
#5.overview
#6.Cast
#7.crew

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.isnull().sum()#to check if any data is missing
movies.dropna(inplace=True)# to remove the missing data
movies.duplicated().sum()#to check if any duplicates are present

movies.iloc[0].genres
#'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'
#['Action','Adventure','Fantasy','Scifi'] convert the above line into this format this is called preprocressing

#this is used to convert the whole string into the list so that the below function can be executed properly
import ast
ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

def convert3(obj):
    L = []
    count=0
    for i in ast.literal_eval(obj):
        if count != 3:
            L.append(i['name'])
            count+=1
        else:
            break
    return L

def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x:x.split()) #this is used to convert string to list
movies.head()


#now there is problem here before merging genres,keywords,cast,crew we have to remove spaces from the strings so that our model doesnt get confused
#for example here 'Sam Worthington' the model will take 'sam' as one entity and 'Worthington' as another but there's another guy called 'Sam mendes' same thing will happen and model will get confused
#so we need to create a function to remove the space and merge the words into one entity
#this is used to remove blank space (lambda x:[i.replace(" ","") for i in x])

movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
movies.head()

#now merge these genres,keywords,cast,crew
movies['Tags'] = movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']
movies.head()

#this is used to create a new table 
new_df = movies[['movie_id','title','Tags']]
#convert tags list to string -> apply(lambda x:" ".join(x))
new_df['Tags'] = new_df['Tags'].apply(lambda x:" ".join(x))
new_df.head()
#converting everything into lower case -> .apply(lambda x:x.lower())
new_df['Tags'] = new_df['Tags'].apply(lambda x:x.lower())
new_df.head()


#Vectorization


#concept of stemming 
#[loved,loving,love] -> [love,love,love]
#[danced,dancing,dance] -> [dance,dance,dance]
#purpose of Nltk => Text Preprocessing: Breaking down text into smaller units via tokenization (words and sentences), removing stopwords, and normalizing data through stemming and lemmatization

import nltk 
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['Tags'] = new_df['Tags'].apply(stem)
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')
vectors = cv.fit_transform(new_df['Tags']).toarray() #(4806,5000) 4806 movies, 5000 words
cv.get_feature_names_out()

from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors) #this function gives the distance between the first movie to all the movies present

sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])
#there is a problem while sorting the distances the problem is while sorting the index postions are change for that we need to preserve the index position
#enumerate this is used to preserve the index positions and we are gonna put them in a list and while sorting we need to sort according to the distances not the indexes
#new_df[new_df['title']=='Batman Begins'].index[0] => this is used to fetch the index of the movies

def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6] #[1:6] is used for fetching first 5 movies
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

recommend('Batman Begins')

import pickle
pickle.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))











 





The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf
